# Week 3
First some exploration to see if we can shrink the parquet without losing `user_id` signal

In [10]:
import duckdb
from datetime import datetime
import colornames
from collections import defaultdict

PARQUET_PATH = "../2022_place_canvas_history_uid.parquet"
# PARQUET_PATH = "../test.parquet"
START_TIME = datetime(int(2022), int(4), int(1), int(12))
END_TIME = datetime(int(2022), int(4), int(1), int(18))

In [ ]:
COLOR_NAME_MAPPING = defaultdict(str)
query = f"""
    SELECT
        DISTINCT pixel_color
    FROM
        '{PARQUET_PATH}'
"""

colors = duckdb.query(query).fetchall()

for c in colors:
    color_name = colornames.find(c[0])
    COLOR_NAME_MAPPING[c[0]] = color_name

In [ ]:
query = f"""
    SELECT
        timestamp,
        coordinate
    FROM
        '{PARQUET_PATH}'
    WHERE
        LENGTH(coordinate) > 9
"""

duckdb.query(query)

# Rank Colors By Distinct Users
Report a ranking of colors by the number of distinct users who placed those during the specified timeframe.

In [21]:
query = f"""
    SELECT
        pixel_color,
        COUNT(DISTINCT user_id) AS user_count
    FROM
        '{PARQUET_PATH}'
    WHERE
        timestamp >= '{START_TIME}' AND
        timestamp <= '{END_TIME}'
    GROUP BY
        pixel_color
    ORDER BY
        user_count DESC
"""

res = duckdb.query(query).fetchall()

for i, tup in enumerate(res):
    hex_color = tup[0]
    color_name = colornames.find(hex_color)

    print(f"{i+1} : {color_name}")

1 : Black
2 : Vermilion
3 : White
4 : Azure
5 : Bright Sun
6 : Turquoise Blue
7 : Pink Salmon
8 : Screamin' Green
9 : Green Haze
10 : Violet Eggplant
11 : Yellow Sea
12 : Curious Blue
13 : Iron
14 : Oslo Gray
15 : Kumera
16 : Amethyst


# Calculate Average Session Length
A session is in progress until a 15-minute window of inactivity is reached. A user must have placed more than one pixel to have had a session.

Return the average session length in seconds during the specified timeframe.

In [44]:
query = f"""
    WITH
        time_diff AS (
            SELECT
                user_id,
                timestamp,
                EPOCH(timestamp) - LAG(EPOCH(timestamp)) OVER (PARTITION BY user_id ORDER BY timestamp ASC) AS time_since_last_placement
            FROM
                '{PARQUET_PATH}'
            WHERE
                timestamp >= '{START_TIME}' AND
                timestamp <= '{END_TIME}'
            GROUP BY
                user_id, timestamp
            ORDER BY
                user_id, timestamp ASC
        )
    SELECT
        *
    FROM
        time_diff

"""

duckdb.query(query)

┌─────────┬─────────────────────────┬───────────────────────────┐
│ user_id │        timestamp        │ time_since_last_placement │
│  int32  │        timestamp        │          double           │
├─────────┼─────────────────────────┼───────────────────────────┤
│      14 │ 2022-04-01 14:24:12.832 │                      NULL │
│      14 │ 2022-04-01 14:31:53.437 │         460.6050000190735 │
│      30 │ 2022-04-01 13:43:06.255 │                      NULL │
│      35 │ 2022-04-01 15:48:55.262 │                      NULL │
│      38 │ 2022-04-01 16:57:29.923 │                      NULL │
│      38 │ 2022-04-01 17:42:57.509 │        2727.5859999656677 │
│      43 │ 2022-04-01 16:20:05.456 │                      NULL │
│      51 │ 2022-04-01 16:25:20.355 │                      NULL │
│      51 │ 2022-04-01 16:32:04.714 │         404.3589999675751 │
│      51 │ 2022-04-01 16:37:26.647 │        321.93300008773804 │
│       · │            ·            │                 ·         │
│       · 

# Pixel Placement Percentiles
Calculate the 50th, 75th, 90th, and 99th percentiles of the number of pixels placed by users during the specified timeframe.

In [17]:
query = f"""
    WITH
        dist AS (
            SELECT
                user_id,
                COUNT(*) AS freq
            FROM
                '{PARQUET_PATH}'
            WHERE
                timestamp >= '{START_TIME}' AND
                timestamp <= '{END_TIME}'
            GROUP BY
                user_id
        )
    SELECT
        quantile_cont(freq, 0.50 ORDER BY freq ASC) AS '50_percentile',
        quantile_cont(freq, 0.75 ORDER BY freq ASC) AS '75_percentile',
        quantile_cont(freq, 0.90 ORDER BY freq ASC) AS '90_percentile',
        quantile_cont(freq, 0.99 ORDER BY freq ASC) AS '99_percentile'
    FROM
        dist
"""
duckdb.query(query)

┌───────────────┬───────────────┬───────────────┬───────────────┐
│ 50_percentile │ 75_percentile │ 90_percentile │ 99_percentile │
│    double     │    double     │    double     │    double     │
├───────────────┼───────────────┼───────────────┼───────────────┤
│           2.0 │           4.0 │          11.0 │          33.0 │
└───────────────┴───────────────┴───────────────┴───────────────┘

# Count First-Time Users
Count how many users placed their first pixel ever within the specified timeframe.

In [18]:
START_TIME = datetime(int(2022), int(4), int(1), int(12))
END_TIME = datetime(int(2022), int(4), int(1), int(18))

query = f"""
    WITH
        during_timeframe AS (
            SELECT
                DISTINCT user_id
            FROM
                '{PARQUET_PATH}'
            WHERE
                timestamp BETWEEN '{START_TIME}' AND '{END_TIME}'
        ),
        before_timeframe AS (
            SELECT
                DISTINCT user_id
            FROM
                '{PARQUET_PATH}'
            WHERE
                timestamp < '{START_TIME}'
        )
    SELECT
        count(*) AS new_users
    FROM
        before_timeframe AS btf
    RIGHT JOIN during_timeframe AS dtf ON
        btf.user_id = dtf.user_id
    WHERE
        btf.user_id IS NULL
"""
duckdb.query(query)

┌───────────┐
│ new_users │
│   int64   │
├───────────┤
│   1062702 │
└───────────┘